# 01 — Exploratory Data Analysis

> **The question that started this project:** Can economic conditions, weather, and supply-chain variables meaningfully predict food prices — and if so, which variables actually matter?

This notebook is where I figure that out before touching any models. Nothing is assumed. Everything gets checked.

---


## Why This Dataset Exists

No single real-world source had all the features we needed. FAO price data exists, but it doesn't come bundled with crop yield, local weather, and inflation in one clean table at daily resolution. So we built the dataset ourselves — 14 years (2010–2023) of daily observations for 10 common food items, simulating how each variable would realistically interact.

This was the hardest part of the project. Every feature range had to be justified to our professor. Why is Exchange_Rate capped at 1.0? (We modeled it as a normalized ratio, not raw JPY or EUR.) Why is Demand_Index between 0.5 and 1.5? (It's a multiplier — 1.0 is baseline, 1.5 represents a 50% demand surge like a festive season.) The dataset design forced us to understand the actual economics before writing any code.

**Dataset summary:**
- **Rows:** 1,000 observations
- **Features:** 16 (Time × 4, Economic × 3, Agricultural × 3, Weather × 3, Market × 3)
- **Target:** `Food_Price` in USD (range: \$0.97 – \$10.75)
- **Missing values:** 0 (by design — synthetic generation doesn't produce gaps)


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 130

ROOT = Path("..")
df = pd.read_csv(ROOT / "data" / "raw" / "food_price_prediction_dataset.csv")
print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:")
print(df.dtypes)


## Data Shape and Basic Health Check

In [ ]:
print("First 5 rows:")
df.head()


In [ ]:
print("Descriptive statistics:")
df.describe().round(2)


In [ ]:
# Zero missing values — expected for synthetic data, but always verify
print(f"Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal nulls: {df.isnull().sum().sum()}")


## Feature Ranges — What Each Variable Represents

Before diving into distributions, here's the design rationale for every feature. These weren't chosen arbitrarily.

| Feature | Range | Why This Range |
|---|---|---|
| Year | 2010–2023 | 14 years captures multiple economic cycles (2008 recovery, COVID shock) |
| Month | 1–12 | Full seasonal coverage |
| Day | 1–28 | Capped at 28 to avoid month-length complexity |
| Season | Spring/Summer/Autumn/Winter | Categorical encoding of growing conditions |
| Inflation_Rate | 1.0–10.98% | Covers stable economies (US ~2%) to high-inflation contexts (Argentina 10%+) |
| Exchange_Rate | 0.1–1.0 | Normalized ratio; extremes simulate weak/strong currency environments |
| GDP_Growth | -2.0 to +5.98% | Includes mild recession (negative) and strong growth |
| Crop_Yield | 2.03–10.0 tons/ha | Range covers drought years (low) to bumper harvests |
| Fertilizer_Price | \$20–\$100/ton | Reflects global commodity price variance |
| Water_Availability | 50–100 index | 50 = stressed supply, 100 = abundant |
| Temperature | -5°C to 39.86°C | Full continental range — affects crop stress |
| Rainfall | 0.07–299.85 mm | Near-drought to heavy monsoon |
| Humidity | 20–90% | Desert dry to tropical humid |
| Transport_Cost | \$50–\$500/ton | Reflects fuel, distance, infrastructure variation |
| Demand_Index | 0.5–1.5 | Multiplier: 0.5 = crisis drop, 1.5 = festive surge |
| Previous_Price | \$1.01–\$9.98 | Yesterday's price — the strongest single predictor |


## Price Distribution

The first thing I want to know about the target: is it normally distributed, skewed, bimodal? The answer affects model choice and evaluation metric selection.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram + KDE
sns.histplot(df["Food_Price"], bins=35, kde=True, color="#4C72B0", ax=axes[0])
axes[0].axvline(df["Food_Price"].mean(), color="#DD4444", ls="--", lw=1.8,
                label=f"Mean: ${df['Food_Price'].mean():.2f}")
axes[0].axvline(df["Food_Price"].median(), color="#44AA44", ls="--", lw=1.8,
                label=f"Median: ${df['Food_Price'].median():.2f}")
axes[0].set_title("Price is roughly uniform — no strong clustering", fontsize=11)
axes[0].set_xlabel("Food Price (USD)"); axes[0].legend()

# Boxplot
sns.boxplot(x=df["Food_Price"], color="#4C72B0", ax=axes[1])
axes[1].set_title(f"Min: ${df['Food_Price'].min():.2f}  Max: ${df['Food_Price'].max():.2f}  Std: ${df['Food_Price'].std():.2f}", fontsize=10)
axes[1].set_xlabel("Food Price (USD)")

plt.suptitle("Price distribution — uniform spread, slight right tail, no extreme outliers", y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(ROOT / "outputs/figures/01_price_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Skewness: {df['Food_Price'].skew():.3f}")
print(f"Kurtosis: {df['Food_Price'].kurtosis():.3f}")


**What this tells me:** The price distribution is close to uniform with a slight right tail. No extreme outliers that would distort RMSE. Skewness near zero means I don't need to log-transform the target — a linear scale works fine.


## Seasonal Patterns

Do seasons meaningfully predict price? My intuition going in was yes — harvest seasons should drop prices, festive periods should push them up.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

season_order = ["Spring", "Summer", "Autumn", "Winter"]
season_stats = df.groupby("Season")["Food_Price"].agg(["mean","std"]).reindex(season_order).reset_index()

axes[0].bar(season_stats["Season"], season_stats["mean"],
            yerr=season_stats["std"], capsize=6,
            color=sns.color_palette("muted", 4), alpha=0.85, edgecolor="black", lw=0.6)
for i, (_, row) in enumerate(season_stats.iterrows()):
    axes[0].text(i, row["mean"]+0.15, f"${row['mean']:.2f}", ha="center", fontsize=10)
axes[0].set_title("Mean ± Std by season — differences are tiny", fontsize=11)
axes[0].set_ylabel("Mean Food Price (USD)"); axes[0].set_ylim(0, 9)

sns.boxplot(data=df, x="Season", y="Food_Price", order=season_order,
            palette="muted", ax=axes[1])
axes[1].set_title("Full distribution by season — nearly identical", fontsize=11)

plt.suptitle("Seasons don't drive price much — the signal is in supply & demand, not the calendar", y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(ROOT / "outputs/figures/02_price_by_season.png", dpi=150, bbox_inches="tight")
plt.show()


**This surprised me.** Spring average (\$5.65) is only marginally higher than Autumn (\$5.28). The error bars overlap entirely. Season as a standalone predictor is nearly useless — but it still goes into the model because it encodes information that interacts with Temperature and Rainfall.


## Year-over-Year Trends

If there's a strong upward price trend over 14 years, ARIMA might actually perform — stationary series are where it lives.


In [ ]:
yr = df.groupby("Year")["Food_Price"].agg(["mean","std"]).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(yr["Year"], yr["mean"], marker="o", lw=2.2, color="#4C72B0", ms=7, label="Mean price")
ax.fill_between(yr["Year"], yr["mean"]-yr["std"], yr["mean"]+yr["std"],
                alpha=0.15, color="#4C72B0", label="±1 std")
ax.axvline(2019, color="#DD4444", ls="--", lw=1.4, label="2019 peak ($6.19 avg)")
ax.set_title("No strong trend — prices fluctuate but don't drift consistently upward", fontsize=11, pad=12)
ax.set_xlabel("Year"); ax.set_ylabel("Mean Food Price (USD)")
ax.set_xticks(yr["Year"]); ax.legend()
plt.tight_layout()
plt.savefig(ROOT / "outputs/figures/03_price_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

print("Year-over-year price means:")
print(yr.set_index("Year")["mean"].round(2))


**Important observation:** There's no persistent upward drift. The series fluctuates around ~\$5.50 with a 2019 spike (\$6.19) and a 2020 dip (\$5.03). This near-stationarity is exactly what ARIMA handles — but the problem is ARIMA can't use any of the 16 other features. It only sees the price series itself.


## Correlations with the Target

The correlation analysis is where things get interesting. What actually moves food prices?


In [ ]:
corr_target = df.corr(numeric_only=True)["Food_Price"].sort_values(ascending=False)
print("Correlation with Food_Price:")
print(corr_target.round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#4CAF50" if v > 0 else "#DD4444" for v in corr_target.drop("Food_Price")]
ax.barh(corr_target.drop("Food_Price").index[::-1],
        corr_target.drop("Food_Price").values[::-1],
        color=colors[::-1], edgecolor="black", lw=0.5)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Previous_Price is almost perfectly correlated (0.99) — everything else is noise at the linear level", fontsize=10, pad=12)
ax.set_xlabel("Pearson Correlation with Food_Price")
plt.tight_layout()
plt.savefig(ROOT / "outputs/figures/04b_correlation_bar.png", dpi=150, bbox_inches="tight")
plt.show()


**The 0.991 correlation between Previous_Price and Food_Price is the single most important finding in this entire EDA.** It explains why XGBoost will dominate — it can use this lag structure directly. It also explains ARIMA's failure: ARIMA builds its own internal autoregressive structure, but when we explicitly give the model `Previous_Price` as a feature, XGBoost captures the same lag dynamics plus everything else.

No other raw feature clears even ±0.07 linear correlation. This doesn't mean they're irrelevant — it means their effects are non-linear, which is exactly where XGBoost's tree structure excels over linear approaches.


In [ ]:
# Full correlation heatmap — engineered features included
df_eng = pd.read_csv(ROOT / "data" / "processed" / "engineered_dataset.csv")
corr_full = df_eng.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(20, 15))
import numpy as np
mask = np.triu(np.ones_like(corr_full, dtype=bool))
sns.heatmap(corr_full, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.4, ax=ax, annot_kws={"size":6.5})
ax.set_title("Post-engineering correlation matrix — interaction terms add new signal", fontsize=12, pad=14)
plt.tight_layout()
plt.savefig(ROOT / "outputs/figures/04_correlation_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()


## Key EDA Findings

Five things I'm taking into modeling:

1. **Previous_Price at r=0.991** is the dominant predictor. Any model that can use it will get very high R². The real question is how well the other features fill in the remaining variance.

2. **No linear signal from weather or economic variables** — individually, Temperature, Rainfall, GDP_Growth all have <0.07 correlation with price. They need to interact with each other and with supply variables to show up. This justifies the interaction-term feature engineering.

3. **Seasons don't separate prices** — Season as a standalone predictor explains almost nothing. It will be one-hot encoded as a control variable, not a primary signal.

4. **No trend in the series** — year-over-year prices don't drift upward, which means ARIMA won't get a "free ride" from a trend component. The variance comes from multi-variable interactions, not temporal drift.

5. **Uniform price distribution** — no extreme skew, no need for target transformation. RMSE and MAE are directly interpretable in USD.
